# Data Preparation

In [242]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

con = sqlite3.connect("../data/hockey.db")
q = lambda sql: pd.read_sql(sql, con)
sql_file = lambda name: q(open(f"../sql/{name}").read())

sns.set_theme(style="whitegrid")

## Data Integration

In this section we join tables so that each row carries information from more than one source. The rubric asks for two things, and we document both:

- **Matching entities between sources**: the key used to link two tables.
- **Conversion of formats**: turning raw fields into values a model can use.

### Coach profile

Query: [`sql/coach_profile.sql`](../sql/coach_profile.sql). One row per coach stint (`coachID`, `year`, `tmID`, `lgID`, `stint`), for **all leagues**, with the person's details from `Master`.

| Source | Target | Key | What we bring in |
|---|---|---|---|
| `Coaches` | `Master` | `coachID` | birth date, `playerID`, `firstNHL` |

- We use a **left join** with `Coaches` as the driving table, so every coaching stint is kept even if the person were missing from `Master`.
- The result has **1,741 rows**, the same as `Coaches`. Every coach finds a match, and no `coachID` appears twice in `Master`, so the join neither loses nor duplicates rows.
- We keep **all leagues** (NHL, WHA, NHA, PCHA, WCHL), not only the NHL, so that a coach's experience can later include seasons coached outside the NHL. `lgID` is kept so we can filter to the NHL when building targets, and to join with `Teams` later.

#### Age at season start (format conversion)

`Master` stores the birth date in three separate columns (`birthYear`, `birthMon`, `birthDay`). We convert them into one number: the coach's **age at the start of the season**.

- **Reference date:** 1 August of `year`. In this dataset `year` is the season's starting year (2005 = 2005-06), so 1 August is before the season begins, and the age is known at prediction time.
- **Birthday rule:** if the birthday falls after 1 August, the coach has not turned the new age yet, so we subtract one year.
- **Missing month or day:** we assume 1 August, the same as the reference date. The age then becomes simply `year − birthYear`, which is wrong by at most one year.
- **Missing year:** the age is left missing (`<NA>`). It is not imputed here; that is handled in the *Missing values* section.

In [243]:
import datetime

coaches = sql_file("coach_profile.sql")

def calculate_age(row):
    if pd.isna(row["birthYear"]):
        return pd.NA
    
    birth_year = int(row["birthYear"])
    birth_month = 8 if pd.isna(row["birthMon"]) else int(row["birthMon"])
    birth_day = 1 if pd.isna(row["birthDay"]) else int(row["birthDay"])
    
    birth_date = datetime.date(
        birth_year,
        birth_month,
        birth_day
    )

    reference_date = datetime.date(
        int(row["year"]),
        8,
        1
    )

    age = reference_date.year - birth_date.year

    if (birth_date.month, birth_date.day) > (
        reference_date.month,
        reference_date.day
    ):
        age -= 1

    return age

coaches["age"] = coaches.apply(calculate_age, axis=1)
coaches["age"] = coaches["age"].astype("Int64")

#### Former player (format conversion)

`Master` only tells us *whether* a coach has a player record. We turn that into two 0/1 features:

| Feature | Rule | Meaning |
|---|---|---|
| `former_player` | `playerID` is not null | played in any league in the data |
| `former_nhl_player` | `firstNHL` is not null | played in the NHL |

Here a null does **not** mean "unknown": it means the person never played. So these flags have no missing values.

After building the features we drop the raw columns (`birthYear`, `birthMon`, `birthDay`, `playerID`, `firstNHL`), which are not useful to a model as they are.

In [244]:
import numpy as np
coaches["former_player"] = np.where(
    coaches["playerID"].isna(), 0, 1
)
coaches["former_nhl_player"] = np.where(
    coaches["firstNHL"].isna(), 0, 1
)

In [245]:
df = coaches.drop(["birthYear", "birthMon", "birthDay", "playerID", "firstNHL"], axis = 1)

#### Checks on the coach profile

In [246]:
younger_coach = df["age"].min()
oldest_coach = df["age"].max()
print(f'The youngest coach as age: {younger_coach}. The Oldest coach as age: {oldest_coach}')

The youngest coach as age: 22. The Oldest coach as age: 74


In [247]:
no_age = df["age"].isnull().sum()
print(f'There is {no_age} missing values')

null_age = df[df["age"].isnull()]
grouped_no_age = null_age.groupby('lgID').size().reset_index(name="count").sort_values("count",ascending=False)

print(grouped_no_age)

There is 60 missing values
   lgID  count
1   NHL     25
0   NHA     19
3  WCHL     10
2  PCHA      3
4   WHA      3


In [248]:
total = df["former_player"].count()
former_player = (df["former_player"] == 1).sum()

percentage = former_player / total * 100

print(f'The percentage of coaches that were former players is {percentage:.2f}%')

total = df["former_nhl_player"].count()
former_nhl_player = (df["former_nhl_player"] == 1).sum()

percentage = former_nhl_player / total * 100

print(f'The percentage of coaches that were former players is {percentage:.2f}%')


The percentage of coaches that were former players is 65.54%
The percentage of coaches that were former players is 63.12%


**Results**

- **Age range: 22 to 74.** Both are real cases, not data errors. The youngest is a player-coach in the NHA in 1916, when teams often had a player also coaching. The oldest is Al Arbour, who came back for one game with the Islanders in 2007. The youngest coach in the NHL itself is 25 (1979). No age is impossible, so no value is removed (validity).
- **60 stints (3.4%) have no age**, from 27 coaches with no `birthYear`. They come mostly from the early leagues (NHA, PCHA, WCHL). The 25 NHL rows are all from before 1940, so this gap should almost disappear after the era selection. We re-check it there (completeness).
- **Missing day or month only:** 30 more stints have a birth year but no month or day; their age uses the approximation described above.
- **Former players:** 65.5% of coaching stints are by former players, and 63.1% by former NHL players. These percentages count **stints (rows)**, not distinct coaches, so coaches with long careers count more than once.

### Coach History

Career features for each coach, built from [`sql/coach_history.sql`](../sql/coach_history.sql): one row per coach per season (all leagues), with the season's games, wins, losses, ties and playoff games summed over all the coach's stints in that season.

**Rule: only seasons before *t*.** Predictions are made the day before season *t* starts, so a coach's history for season *t* may only use seasons up to *t-1*. Anything from season *t* itself (its wins, its playoff games) is not known yet and would be leakage.

| Source | Key | What we bring in |
|---|---|---|
| `Coaches` (aggregated per season) | `coachID` | games, wins, ties and playoff games of every earlier season |

#### Sort by coach and year

The running totals below are computed in row order, so each coach's seasons must be in chronological order first.

In [249]:
history = sql_file("coach_history.sql")

history = history.sort_values(by=["coachID", "year"], ascending=[True, True])

print(history.head())

     coachID  year lgID  n_stints     g     w     l     t  postg
0  abelsi01c  1952  NHL         1  70.0  27.0  28.0  15.0    7.0
1  abelsi01c  1953  NHL         1  70.0  12.0  51.0   7.0    0.0
2  abelsi01c  1957  NHL         1  33.0  16.0  12.0   5.0    4.0
3  abelsi01c  1958  NHL         1  70.0  25.0  37.0   8.0    0.0
4  abelsi01c  1959  NHL         1  70.0  26.0  29.0  15.0    6.0


#### How a tie counts

A tie counts as **half a win**: win% = (wins + ½·ties) / games. This matches how hockey standings value a tie (1 point out of 2) and is consistent with `pts_pct` in the team features. It is expressed as a fraction (0–1), the same scale as `pts_pct`.

In [250]:
history["win_pct"] = (history["w"] + 0.5 * history["t"])/history["g"]

print(history.head())

     coachID  year lgID  n_stints     g     w     l     t  postg   win_pct
0  abelsi01c  1952  NHL         1  70.0  27.0  28.0  15.0    7.0  0.492857
1  abelsi01c  1953  NHL         1  70.0  12.0  51.0   7.0    0.0  0.221429
2  abelsi01c  1957  NHL         1  33.0  16.0  12.0   5.0    4.0  0.560606
3  abelsi01c  1958  NHL         1  70.0  25.0  37.0   8.0    0.0  0.414286
4  abelsi01c  1959  NHL         1  70.0  26.0  29.0  15.0    6.0  0.478571


#### Running totals (up to and including season *t*)

For each coach, a cumulative sum over his seasons in chronological order of games, wins, ties, playoff games and number of seasons. The sum is computed **within each coach** (grouped by `coachID`), so one coach's career never carries over into the next coach's rows.

At this point the totals still **include** season *t*, so they cannot be used as features yet.

In [251]:
count_columns = ["g", "w", "t", "l", "postg", "seasons_before"]

history["seasons_before"] = 1

career_columns = [
    "games_before",
    "wins_before",
    "ties_before",
    "losses_before",
    "postgames_before",
    "seasons_before"
]

history[career_columns] = history.groupby("coachID")[count_columns].cumsum()

print(history.head())

     coachID  year lgID  n_stints     g     w     l     t  postg   win_pct  \
0  abelsi01c  1952  NHL         1  70.0  27.0  28.0  15.0    7.0  0.492857   
1  abelsi01c  1953  NHL         1  70.0  12.0  51.0   7.0    0.0  0.221429   
2  abelsi01c  1957  NHL         1  33.0  16.0  12.0   5.0    4.0  0.560606   
3  abelsi01c  1958  NHL         1  70.0  25.0  37.0   8.0    0.0  0.414286   
4  abelsi01c  1959  NHL         1  70.0  26.0  29.0  15.0    6.0  0.478571   

   seasons_before  games_before  wins_before  ties_before  losses_before  \
0               1          70.0         27.0         15.0           28.0   
1               2         140.0         39.0         22.0           79.0   
2               3         173.0         55.0         27.0           91.0   
3               4         243.0         80.0         35.0          128.0   
4               5         313.0        106.0         50.0          157.0   

   postgames_before  
0               7.0  
1               7.0  
2       

#### Move the totals to "before season *t*"

Each running total is shifted down by one season within the same coach, so the row for season *t* holds the totals **up to season *t-1***. Career totals do not need consecutive seasons: if a coach skipped years, the shifted value is still the sum of all his earlier seasons.

**First season of each coach (rookie coach).** After the shift there is nothing before it, but the meaning differs per feature:

| Feature | First season | Why |
|---|---|---|
| seasons, games, wins, ties, playoff games before *t* | **0** | the coach really had zero |
| `win_career_pct` | **missing** | 0 / 0 is undefined; a rookie has no record, which is not the same as a 0% record. Handled in *Missing values* |
| `is_rookie_coach` | **1** | flags the rows where the career win% is missing for this reason |

In [252]:
history[career_columns] = history.groupby("coachID")[career_columns].shift(1).fillna(0.0)



history["win_before_pct"] = (history["wins_before"] + 0.5 * history["ties_before"])/history["games_before"]
history["is_rookie_coach"] = np.where(history["games_before"] == 0.0, 1, 0)
print(history.head())

     coachID  year lgID  n_stints     g     w     l     t  postg   win_pct  \
0  abelsi01c  1952  NHL         1  70.0  27.0  28.0  15.0    7.0  0.492857   
1  abelsi01c  1953  NHL         1  70.0  12.0  51.0   7.0    0.0  0.221429   
2  abelsi01c  1957  NHL         1  33.0  16.0  12.0   5.0    4.0  0.560606   
3  abelsi01c  1958  NHL         1  70.0  25.0  37.0   8.0    0.0  0.414286   
4  abelsi01c  1959  NHL         1  70.0  26.0  29.0  15.0    6.0  0.478571   

   seasons_before  games_before  wins_before  ties_before  losses_before  \
0             0.0           0.0          0.0          0.0            0.0   
1             1.0          70.0         27.0         15.0           28.0   
2             2.0         140.0         39.0         22.0           79.0   
3             3.0         173.0         55.0         27.0           91.0   
4             4.0         243.0         80.0         35.0          128.0   

   postgames_before  win_before_pct  is_rookie_coach  
0               0.0

#### Clean up

We drop the season's own results (`g`, `w`, `l`, `t`, `postg`, `win_pct`). They describe season *t* itself, so keeping them as features would be leakage. They are still available from the SQL if needed for the targets.

In [253]:
history = history.drop(["g", "w", "l", "t", "postg", "win_pct", "n_stints"], axis = 1)

print(history.head())

     coachID  year lgID  seasons_before  games_before  wins_before  \
0  abelsi01c  1952  NHL             0.0           0.0          0.0   
1  abelsi01c  1953  NHL             1.0          70.0         27.0   
2  abelsi01c  1957  NHL             2.0         140.0         39.0   
3  abelsi01c  1958  NHL             3.0         173.0         55.0   
4  abelsi01c  1959  NHL             4.0         243.0         80.0   

   ties_before  losses_before  postgames_before  win_before_pct  \
0          0.0            0.0               0.0             NaN   
1         15.0           28.0               7.0        0.492857   
2         22.0           79.0               7.0        0.357143   
3         27.0           91.0              11.0        0.395954   
4         35.0          128.0              11.0        0.401235   

   is_rookie_coach  
0                1  
1                0  
2                0  
3                0  
4                0  


#### Result

One row per (coach, season) with the coach's career up to the previous season. It is not attached to teams yet: that happens when building the targets, where we choose which coach describes each team-season.

In [254]:
is_rookie = (history["is_rookie_coach"] == 1).sum()

print(f'There is {is_rookie} values')

print(history.head())

There is 384 values
     coachID  year lgID  seasons_before  games_before  wins_before  \
0  abelsi01c  1952  NHL             0.0           0.0          0.0   
1  abelsi01c  1953  NHL             1.0          70.0         27.0   
2  abelsi01c  1957  NHL             2.0         140.0         39.0   
3  abelsi01c  1958  NHL             3.0         173.0         55.0   
4  abelsi01c  1959  NHL             4.0         243.0         80.0   

   ties_before  losses_before  postgames_before  win_before_pct  \
0          0.0            0.0               0.0             NaN   
1         15.0           28.0               7.0        0.492857   
2         22.0           79.0               7.0        0.357143   
3         27.0           91.0              11.0        0.395954   
4         35.0          128.0              11.0        0.401235   

   is_rookie_coach  
0                1  
1                0  
2                0  
3                0  
4                0  


## Data Quality

## Era Selection

## Targets

## Feature engineering

## Missing values

## Outliers

## Redundancy

## Save the Final table